# RAG Evaluation Guide

Mental model for why RAG evaluation is structured differently, the RAG Triad in detail, and pointers to deeper notebooks.

## Why RAG Evaluation Is Different

Standard LLM evaluation asks one question: **"Is the output correct?"**

RAG evaluation asks **three separate questions** — one per pipeline component. Each component can fail independently, and a single end-to-end score hides which one is broken.

```
                    ┌─────────────┐
                    │    Query    │
                    └──────┬──────┘
                           │
              ┌────────────▼────────────┐
              │       Retriever         │
              │  (finds context chunks) │
              └────────────┬────────────┘
                           │
         ┌─────────────────┼─────────────────┐
         │                 │                 │
         ▼                 ▼                 ▼
  Context Relevance   Groundedness      Answer Relevance
  ─────────────────   ────────────      ────────────────
  "Does the context   "Is the answer    "Does the answer
   contain what is    only based on     actually address
   needed?"           the context?"     the question?"

  Retriever metric   Generator metric   End-to-end metric
```

**Key insight:** A high end-to-end score can hide a broken retriever — the generator may answer correctly from parametric (training) memory even when retrieval contributed nothing. You only discover this by measuring each leg separately.

## The RAG Triad

### 1. Faithfulness (Groundedness)
**Question:** Is every claim in the answer supported by the retrieved context?

- Decompose the answer into atomic claims
- For each claim: *"Is this supported by the context?"* (yes/no per claim)
- Score = `grounded claims / total claims`
- **Target:** ≥ 0.80 (higher for regulated domains)
- **Low score → Generator problem.** Fix: lower temperature, stricter grounding prompt, citation constraints.

---

### 2. Context Relevance
**Question:** Were the retrieved chunks actually useful for answering the query?

- For each retrieved chunk: *"Does this chunk contain information relevant to the query?"*
- Score = `relevant chunks / total retrieved chunks`
- **Target:** ≥ 0.70
- **Low score → Retriever problem.** Fix: embedding model, chunk size, top-k, re-ranking.

---

### 3. Answer Relevance
**Question:** Does the answer actually address what the user asked?

- Measures semantic alignment between question and answer — a faithful answer can still be off-topic
- Score = cosine similarity between question embedding and back-generated questions from the answer
- **Target:** ≥ 0.80
- **Low score → Alignment problem.** Fix: system prompt, query reformulation.

---

### Diagnosis Map

| Context Relevance | Faithfulness | Answer Relevance | Root Cause | Fix |
|:-:|:-:|:-:|---|---|
| Low | Any | Any | Retriever — wrong chunks | Embedding model, chunk size, top-k |
| High | Low | Any | Generator — hallucinating | Prompt constraints, temperature |
| High | High | Low | Alignment — off-topic answer | System prompt, query rewriting |
| High | High | High | ✓ Working correctly | — |

## Hallucination vs Context Neglect

Both produce wrong answers but have different root causes and different fixes:

```
Hallucination    → Model INVENTS facts not present in the context
                   Detected by: faithfulness score < 0.8
                   Fix: lower temperature, stricter grounding prompt

Context Neglect  → Model IGNORES the context and answers from training memory
                   Detected by: run query with and without context —
                                if answer is identical, retrieval is being ignored
                   Fix: "Answer ONLY using the context below" in system prompt
```

## What to Read Next

| Topic | Notebook |
|-------|----------|
| Retrieval ranking metrics (MRR, NDCG, Precision@K, Recall@K) | `rag_evaluation_metrics.ipynb` |
| Context window realism — why eval context must match prod | `rag_evaluation_metrics.ipynb` |
| RAGAS, TruLens, DeepEval — code examples | `rag_evaluation_frameworks.ipynb` |
| LangSmith, Langfuse, Arize Phoenix — tool comparisons | `rag_evaluation_tools.ipynb` |

## Relationship to Agent Evaluation

RAG evaluation is a **subset** of agent evaluation — it applies whenever an agent retrieves context before generating an answer. The two sets of docs are complementary:

| Agent Evaluation (`../agents/`) | RAG Evaluation (this folder) |
|---------------------------------|------------------------------|
| Why agent eval is different | Why RAG eval is different from output-only eval |
| Component error rates | Per-component RAG Triad diagnosis |
| LLM-as-judge patterns | Faithfulness / groundedness (QAG pattern) |
| Trajectory metrics | Retrieval ranking metrics (MRR, NDCG) |
| Production eval architecture | RAG-specific production monitoring signals |

**Start with `../agents/`** if you are new to evaluation entirely.  
**Start here** if you already understand agent eval and need RAG-specific depth.
